# LightGBM Touchdown Delay Classifier (2025)

Predicts whether a KLM flight will experience a touchdown delay using pre-departure and in-flight plan features.

**Target:** Binary — `1` if `fl_touchdown_delay_min > 0`, else `0`  
**Split:** Train = Jan–Oct 2025 | Validation = Nov 2025 | Test = Dec 2025

## 1. Setup & Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path(r"C:\Users\k454047\OneDrive - Air France KLM\Desktop\THESIS\Code Test")

## 2. Load & Clean Data

Load the cleaned flight dataset, filter to **2025 flights only**, and apply the same outlier-removal and quality filters developed during exploration.

In [2]:
# Load the raw KLM dataset
input_path = PROJECT_ROOT / "data" / "raw" / "klm_raw_2023_2026.parquet"
df_raw = pd.read_parquet(input_path)
print("Raw shape:", df_raw.shape)

# ── Parse datetime columns ─────────────────────────────────────────────────
datetime_cols = [
    "fl_scheduled_departure_date_utc",
    "fl_scheduled_arrival_date_utc",
    "fl_scheduled_off_block_time_utc",
    "fl_actual_off_block_time_utc",
    "fl_scheduled_in_block_time_utc",
    "fl_actual_in_block_time_utc",
    "fp_flight_plan_timestamp_utc",
    "fl_estimated_touchdown_time_utc",
    "fl_actual_touchdown_time_utc",
    "fl_prev_leg_sibt_utc",
    "fl_prev_leg_aibt_utc",
]
for col in datetime_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_datetime(df_raw[col], errors="coerce")

# ── Replace fake nulls ─────────────────────────────────────────────────────
obj_cols = df_raw.select_dtypes(include=["object", "string"]).columns
df_raw[obj_cols] = df_raw[obj_cols].replace(["NULL", "None", "", "NaN"], np.nan)
df_raw = df_raw.replace(r"^\s*$", np.nan, regex=True)

# ── Filter: service type J, non-zero flight plan ───────────────────────────
df = df_raw[
    (df_raw["fl_service_type"] == "J") &
    (df_raw["has_flight_plan_data"] != 0)
].copy()

# ── Filter: 2025 only ─────────────────────────────────────────────────────
df = df[df["fl_scheduled_off_block_time_utc"].dt.year == 2025].copy()
print("2025 shape:", df.shape)

TypeError: data type 'dbdate' not understood

In [ ]:
# ── Numeric coercion for key columns ──────────────────────────────────────
numeric_cols = [
    "fl_scheduled_block_time_min",
    "fl_actual_block_time_min",
    "fl_actual_taxi_out_duration_min",
    "fl_flight_plan_trip_duration_min",
    "fl_actual_trip_duration_min",
    "fl_actual_taxi_in_duration_min",
    "fl_departure_delay_difference_min",
    "fl_block_delay_difference_min",
    "fl_arrival_delay_difference_min",
    "fl_trip_duration_deviation_min",
    "fp_flight_plan_version",
    "fl_great_circle_distance_nm",
    "fp_cost_index",
    "fl_prev_leg_arrival_delay_min",
    "fl_touchdown_delay_min",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ── Outlier removal: block/trip bounds ────────────────────────────────────
df = df[
    (df["fl_block_delay_difference_min"].abs() <= 60) &
    (df["fl_trip_duration_deviation_min"].abs() <= 60)
].copy()

# ── Outlier removal: IQR on operational durations ────────────────────────
iqr_cols = [
    "fl_actual_taxi_out_duration_min",
    "fl_actual_taxi_in_duration_min",
    "fl_departure_delay_difference_min",
    "fl_arrival_delay_difference_min",
    "fl_prev_leg_arrival_delay_min",
    "fl_touchdown_delay_min",
]
for col in iqr_cols:
    if col not in df.columns:
        continue
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[
        (df[col] >= q1 - 1.5 * iqr) &
        (df[col] <= q3 + 1.5 * iqr)
    ].copy()

# ── Remove negative taxi durations ────────────────────────────────────────
df = df[
    (df["fl_actual_taxi_in_duration_min"] >= 0) &
    (df["fl_actual_taxi_out_duration_min"] >= 0)
].copy()

# ── Flight plan lead time feature and quality filter ──────────────────────
df["fp_flight_plan_timestamp_utc"] = df["fp_flight_plan_timestamp_utc"].dt.tz_localize(None)
df["fp_lead_time_min"] = (
    (df["fp_flight_plan_timestamp_utc"] - df["fl_scheduled_off_block_time_utc"])
    .dt.total_seconds() / 60
)
# Keep only flights where the flight plan was filed at least 90 min before departure
df = df[df["fp_lead_time_min"] <= -90].copy()

print("Shape after cleaning:", df.shape)
df[["fl_touchdown_delay_min", "fl_block_delay_difference_min", "fp_lead_time_min"]].describe()

## 3. Feature Engineering

Create **categorical** features for hour, day-of-week, and month from all relevant timestamp columns.  
Cyclical (sin/cos) encoding is added alongside the categorical labels so the model can also learn linear cyclical patterns.

In [ ]:
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def add_time_categoricals(df, ts_col, prefix):
    """
    Given a datetime column `ts_col`, add categorical hour / day-of-week / month
    features with the given `prefix`.  Also adds sin/cos cyclic encodings.
    """
    dt = df[ts_col]
    # ── Categorical labels ─────────────────────────────────────────────────
    df[f"{prefix}_hour"]       = pd.Categorical(dt.dt.hour, categories=range(24), ordered=True)
    df[f"{prefix}_dayofweek"]  = pd.Categorical(dt.dt.day_name(), categories=DAY_ORDER, ordered=True)
    df[f"{prefix}_month"]      = pd.Categorical(dt.dt.month, categories=range(1, 13), ordered=True)
    # ── Cyclic encodings ───────────────────────────────────────────────────
    df[f"{prefix}_hour_sin"]   = np.sin(2 * np.pi * dt.dt.hour   / 24)
    df[f"{prefix}_hour_cos"]   = np.cos(2 * np.pi * dt.dt.hour   / 24)
    df[f"{prefix}_month_sin"]  = np.sin(2 * np.pi * dt.dt.month  / 12)
    df[f"{prefix}_month_cos"]  = np.cos(2 * np.pi * dt.dt.month  / 12)
    df[f"{prefix}_dow_sin"]    = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
    df[f"{prefix}_dow_cos"]    = np.cos(2 * np.pi * dt.dt.dayofweek / 7)
    return df

# Scheduled departure (SOBT)
df = add_time_categoricals(df, "fl_scheduled_off_block_time_utc", "dep")

# Scheduled arrival (SIBT)
df = add_time_categoricals(df, "fl_scheduled_in_block_time_utc",  "arr")

# Previous leg actual arrival
df = add_time_categoricals(df, "fl_prev_leg_aibt_utc", "prev_arr")

print("New time features added. Sample:")
df[["dep_hour", "dep_dayofweek", "dep_month",
    "arr_hour", "arr_dayofweek", "arr_month"]].head()

## 4. Create Binary Target Variable

`fl_touchdown_delay_min > 0` → **delayed** (class 1)  
`fl_touchdown_delay_min ≤ 0` → **on-time / early** (class 0)

In [ ]:
# Drop rows where the target is unknown
df = df.dropna(subset=["fl_touchdown_delay_min"]).copy()

# Binary target: 1 = delayed at touchdown, 0 = on-time or early
df["touchdown_delayed"] = (df["fl_touchdown_delay_min"] > 0).astype(int)

class_counts = df["touchdown_delayed"].value_counts()
print("Class distribution:")
print(class_counts)
print(f"\nDelay rate: {class_counts[1] / len(df):.2%}")

plt.figure(figsize=(5, 3))
class_counts.plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["On-time / Early (0)", "Delayed (1)"], rotation=0)
plt.ylabel("Number of flights")
plt.title("Touchdown Delay — Class Distribution")
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Split

| Set | Months | Purpose |
|-----|--------|---------|
| Train | Jan – Oct 2025 | Fit the model |
| Validation | Nov 2025 | Tune hyperparameters / early stopping |
| Test | Dec 2025 | Final unbiased evaluation |

In [ ]:
month = df["fl_scheduled_off_block_time_utc"].dt.month

df_train = df[month <= 10].copy()
df_val   = df[month == 11].copy()
df_test  = df[month == 12].copy()

print(f"Train : {len(df_train):,} flights  (Jan–Oct)")
print(f"Val   : {len(df_val):,}  flights  (Nov)")
print(f"Test  : {len(df_test):,}  flights  (Dec)")
print()
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    rate = split["touchdown_delayed"].mean()
    print(f"{name} delay rate: {rate:.2%}")

## 6. Feature Definition & Preprocessing

Select only **pre-departure / in-flight-plan** features — nothing that would only be known after touchdown.  
LightGBM handles `category` dtype natively, so string/high-cardinality columns are cast accordingly.

In [ ]:
TARGET = "touchdown_delayed"

# ── Numeric features ───────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "fl_scheduled_block_time_min",
    "fl_flight_plan_trip_duration_min",
    "fl_great_circle_distance_nm",
    "fp_cost_index",
    "fp_lead_time_min",
    "fl_prev_leg_arrival_delay_min",
    "fp_flight_plan_version",
    # Cyclic time encodings
    "dep_hour_sin",  "dep_hour_cos",
    "dep_month_sin", "dep_month_cos",
    "dep_dow_sin",   "dep_dow_cos",
    "arr_hour_sin",  "arr_hour_cos",
    "arr_month_sin", "arr_month_cos",
    "arr_dow_sin",   "arr_dow_cos",
]

# ── Categorical features (label → LightGBM handles internally) ────────────
CAT_FEATURES = [
    "dep_hour",         # pd.Categorical, ordered
    "dep_dayofweek",
    "dep_month",
    "arr_hour",
    "arr_dayofweek",
    "arr_month",
    "fl_aircraft_type",
    "fl_flight_product",
    "fl_season",
    "fl_departure_airport",
    "fp_planned_arrival_runway",
    "fl_star",
    "fl_city_pair",
]

ALL_FEATURES = NUMERIC_FEATURES + CAT_FEATURES

# ── Keep only columns that exist in the dataframe ─────────────────────────
ALL_FEATURES = [f for f in ALL_FEATURES if f in df.columns]
CAT_FEATURES = [f for f in CAT_FEATURES if f in df.columns]

print(f"Total features: {len(ALL_FEATURES)}  "
      f"(numeric: {len(NUMERIC_FEATURES)}, categorical: {len(CAT_FEATURES)})")

def prepare_split(split_df):
    X = split_df[ALL_FEATURES].copy()
    # Cast high-cardinality string columns to category
    for col in CAT_FEATURES:
        if col in X.columns:
            X[col] = X[col].astype("category")
    y = split_df[TARGET]
    return X, y

X_train, y_train = prepare_split(df_train)
X_val,   y_val   = prepare_split(df_val)
X_test,  y_test  = prepare_split(df_test)

print(f"\nX_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
X_train.head()

## 7. Train LightGBM Classifier

Early stopping is driven by the **validation AUC**.  
`scale_pos_weight` compensates for class imbalance automatically.

In [ ]:
# Class-imbalance weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.3f}  (neg={neg:,}, pos={pos:,})")

lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_FEATURES, free_raw_data=False)
lgb_val   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_FEATURES, reference=lgb_train, free_raw_data=False)

params = {
    "objective":         "binary",
    "metric":            ["binary_logloss", "auc"],
    "boosting_type":     "gbdt",
    "n_estimators":      1000,
    "learning_rate":     0.05,
    "num_leaves":        63,
    "max_depth":         -1,
    "min_child_samples": 50,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "scale_pos_weight":  scale_pos_weight,
    "random_state":      42,
    "verbose":           -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=callbacks,
)

print(f"\nBest iteration: {model.best_iteration}")

## 8. Evaluate on Validation Set

Threshold is set at **0.5** by default; adjust below if the operational cost of false negatives vs false positives differs.

In [ ]:
THRESHOLD = 0.5

def evaluate(model, X, y, split_name="Val", threshold=THRESHOLD):
    proba = model.predict(X)
    preds = (proba >= threshold).astype(int)

    auc  = roc_auc_score(y, proba)
    ap   = average_precision_score(y, proba)

    print(f"\n{'─'*50}")
    print(f"  {split_name} — AUC-ROC: {auc:.4f}   Avg Precision: {ap:.4f}")
    print(f"{'─'*50}")
    print(classification_report(y, preds, target_names=["On-time/Early", "Delayed"]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"LightGBM Classifier — {split_name} Set", fontsize=13)

    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y, preds, display_labels=["On-time/Early", "Delayed"],
        ax=axes[0], colorbar=False, cmap="Blues"
    )
    axes[0].set_title("Confusion Matrix")

    # ROC curve
    RocCurveDisplay.from_predictions(y, proba, ax=axes[1], name=split_name)
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_title(f"ROC Curve  (AUC = {auc:.3f})")

    # Precision-Recall curve
    precision, recall, _ = precision_recall_curve(y, proba)
    axes[2].plot(recall, precision, lw=1.5)
    axes[2].set_xlabel("Recall")
    axes[2].set_ylabel("Precision")
    axes[2].set_title(f"Precision-Recall  (AP = {ap:.3f})")

    plt.tight_layout()
    plt.show()

evaluate(model, X_val, y_val, split_name="Validation")

## 9. Final Evaluation on Test Set (Dec 2025)

Run **once only** — after all hyperparameter decisions are locked in.

In [ ]:
evaluate(model, X_test, y_test, split_name="Test (Dec 2025)")

## 10. Feature Importance

In [ ]:
importance = (
    pd.Series(model.feature_importance(importance_type="gain"), index=model.feature_name())
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gain-based importance
importance.head(25).plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].invert_yaxis()
axes[0].set_title("Top 25 Features — Gain")
axes[0].set_xlabel("Gain")

# Split-based importance
split_imp = (
    pd.Series(model.feature_importance(importance_type="split"), index=model.feature_name())
    .sort_values(ascending=False)
)
split_imp.head(25).plot(kind="barh", ax=axes[1], color="tomato")
axes[1].invert_yaxis()
axes[1].set_title("Top 25 Features — Split Count")
axes[1].set_xlabel("Split Count")

plt.tight_layout()
plt.show()

print("\nTop 10 features by gain:")
print(importance.head(10))